In [ ]:
!pip install pandas
!pip install numpy
!pip install matplotlib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
finance_data = pd.read_csv('spiff_data-2.csv')


In [ ]:
finance_names = finance_data.keys()[2:].to_list()

In [ ]:
def moving_avarage_cross_over(data_frame, name, short_window=25, long_window=100):
    data_frame = data_frame.copy()
    long_window_text = "_" + str(long_window)
    short_window_text = "_" + str(short_window)
    combine_str = short_window_text + long_window_text

    nan_indicies = data_frame[name][data_frame[name].isna()].index

    nan_min = min(nan_indicies)
    nan_max = max(nan_indicies[nan_indicies < 5255])

    data_frame[name + '_MA_short' + combine_str] = data_frame[name].rolling(window=short_window).mean()
    data_frame[name + '_MA_long' + combine_str] = data_frame[name].rolling(window=long_window).mean()

    data_frame[name + '_signal' + combine_str] = 0
    data_frame.loc[data_frame[name+'_MA_short' + combine_str] >= data_frame[name+'_MA_long' + combine_str], name+'_signal' + combine_str] = 1
    data_frame.loc[data_frame[name+'_MA_short' + combine_str] < data_frame[name+'_MA_long' + combine_str], name+'_signal' + combine_str] = -1
#    data_frame[name+'_position' + combine_str] = data_frame[name+'_signal' + combine_str].diff()
    data_frame = data_frame.drop(columns=[name + '_MA_short' + combine_str,  name + '_MA_long' + combine_str])
    return data_frame

def channal_breakout(data_frame, name, c = 0.5, d = 3, k = 5, x = 0.03, j = 20, combine_str = "_none"):
    data_frame = data_frame.copy()

    high = data_frame[name].shift(d).rolling(window= j).max()
    low = data_frame[name].shift(d).rolling(window= j).min()

    channal_exist = (high <= low * (1 + c))
    upper = low * (1 + c)
    lower = high * (1 - c)

    lag_min = data_frame[name].rolling(window = d).min()
    lag_max = data_frame[name].rolling(window = d).max()

    data_frame[name + '_channal_long' + combine_str] = (lag_min > upper * (1 + x)) & channal_exist
    data_frame[name + '_channal_short' + combine_str] = (lag_max < lower * (1 - x)) & channal_exist

    nan_Frame = data_frame[name].isna()

    bad = nan_Frame.rolling(window=d + j + k, min_periods=1).max()

    data_frame[name + '_valid_break_outs' + combine_str] = bad.eq(0).fillna(False)

    data_frame[name + '_signal' + combine_str] = 0
    for i in range(d + j, len(data_frame)-k):
      if data_frame.loc[i + k, name + '_valid_break_outs' + combine_str]:
        if data_frame.loc[i, name + '_channal_long' + combine_str]:
          data_frame.loc[i: i + k, name + '_signal' + combine_str] = 1
        elif data_frame.loc[i, name + '_channal_short' + combine_str]:
          data_frame.loc[i: i + k, name + '_signal' + combine_str] = - 1
      else:
        data_frame.loc[i, name + '_signal' + combine_str] = 0

    vc = data_frame[name + '_signal' + combine_str].value_counts()

    print(vc.get(-1, 0), vc.get(1, 0))

    data_frame = data_frame.drop(columns=[name + '_channal_short' + combine_str,  name + '_channal_long' + combine_str, name + '_valid_break_outs' + combine_str])
    return data_frame


def remove_1000(data_frame, name):
    data_frame = data_frame.copy()
    mask = data_frame[name] == 1000
    idx = np.where(mask)[0]  # positional indices

    # avoid boundary issues
    valid = (idx > 0) & (idx < len(data_frame) - 1)
    idx = idx[valid]

    before = data_frame[name].iloc[idx - 1].to_numpy()
    after  = data_frame[name].iloc[idx + 1].to_numpy()
    print(before, after)
    data_frame.iloc[idx, data_frame.columns.get_loc(name)] = (before + before) / 2

    return data_frame

def training_test_split(data_frame, part_slit=0.8):
    nan_indicies = data_frame["gurkor"][data_frame[name].isna()].index
    nan_min = 0
    print(nan_indicies)
    if len(nan_indicies[nan_indicies>3000]) ==0:
      nan_max = len(data_frame["gurkor"])
    else:
      nan_max = min(nan_indicies[nan_indicies>3000])-1
    print(nan_min, nan_max)

    split_idx = int(nan_min + (nan_max - nan_min) * part_slit)
    data_frame = data_frame.copy()
    train_df = data_frame.iloc[:split_idx]
    test_df = data_frame.iloc[split_idx:]

    return train_df, test_df

In [ ]:
def get_return_portfolie(df, names, blanka = False, weighting = "equal", lend_possible = True, combine_str="_none"):

  cols = [name + "_signal" + combine_str for name in names]
  for name in names:
    df[name + "_change" + combine_str] = df[name]/df[name].shift(1)

  if blanka:
    df["stocks_active" + combine_str] = (df[cols] == -1 or df[cols]==1).sum(axis=1).shift(1)
  else:
    df["stocks_active" + combine_str] = (df[cols] == 1).sum(axis=1).shift(1)

  return_no_trade = 1.03 ** (1/252) - 1 if lend_possible else 0.

  df["total_return_per_day" + combine_str] = np.nan

  for name in names:
    df[name + "_return" + combine_str] = np.nan

  for i in range(1, len(df)):

    for name in names:
      if blanka:
        df.loc[i, name + "_return" + combine_str] = (df.loc[i, name + "_change" + combine_str] - 1) * (int(df.loc[i - 1, name + "_signal" + combine_str] == 1) +
                                                    (1 - df.loc[i, name + "_change" + combine_str]) * int(df.loc[i - 1][name + "_signal" + combine_str] == -1))
      else:
        df.loc[i, name + "_return" + combine_str] = (df.loc[i, name + "_change" + combine_str] - 1) * int(df.loc[i - 1, name + "_signal" + combine_str] == 1)

    if weighting == "equal":

      if df.loc[i, "stocks_active" + combine_str] == 0 or np.isnan(df.loc[i, "stocks_active" + combine_str]):
        df.loc[i, "total_return_per_day" + combine_str] = return_no_trade + 1

      else:
        stocks_return_names = [name + "_return" + combine_str for name in names]
        returns =  [df.loc[i, stocks_return_name] / df.loc[i, "stocks_active" + combine_str] for stocks_return_name in stocks_return_names]
        df.loc[i, "total_return_per_day" + combine_str] = np.array(returns).sum() + 1

    elif weighting == "active":
      return_weighted = 0
      sum_weighted = 0
      for name in names:
        if blanka or df.loc[i - 1, name + "_signal" + combine_str] == 1:
          return_weighted += df.loc[i, name + "_return" + combine_str] * df.loc[i, name].shift(1)
          sum_weighted += df.loc[i, name].shift(1)

      if sum_weighted == 0:
        df.loc[i, "total_return_per_day" + combine_str] = return_no_trade + 1
      else:
        df.loc[i, "total_return_per_day" + combine_str] = return_weighted / sum_weighted + 1
  df.drop(columns=[name + "_return" + combine_str for name in names])
  df.drop(columns=[name + "_change" + combine_str for name in names])
  df.drop(columns=[name + "_signal" + combine_str for name in names])
  df.drop(columns=["stocks_active" + combine_str for name in names])

  return df

def get_return_portfolie_buy_and_hold(df, names, blanka=False, weighting="equal", lend_possible=True):
    combine_str = "_buy_and_hold"

    # compute individual returns first
    for name in names:
        df[name + "_return" + combine_str] = df[name] / df[name].shift(1)

    return_no_trade = 1.03 ** (1/252) if lend_possible else 1.0

    df["total_return_per_day" + combine_str] = np.nan

    stocks_active = len(names)

    for i in range(1, len(df)):
        stocks_active = df.loc[i, names].notna().sum()

        if weighting == "equal":
            returns = [df.loc[i, name + "_return" + combine_str] / stocks_active for name in names]
            df.loc[i, "total_return_per_day" + combine_str] = np.nansum(returns)

        elif weighting == "active":
            return_weighted = 0.0
            sum_weighted = 0.0

            for name in names:
                w = df.loc[i-1, name]
                r = df.loc[i, name + "_return" + combine_str]

                return_weighted += r * w
                sum_weighted += w

            if sum_weighted == 0:
                df.loc[i, "total_return_per_day" + combine_str] = return_no_trade
            else:
                df.loc[i, "total_return_per_day" + combine_str] = return_weighted / sum_weighted
    df.drop(columns=[name + "_return" + combine_str for name in names])
    return df

def get_sharpe_ratio(df, names, intervall=21, short_window=25, long_window=100, combine_str = None, weighting = "equal", renta = 1.03):
  if combine_str == None:
    combine_str = "_" + str(short_window) + "_" + str(long_window) if combine_str is None else combine_str
    lower_limit = long_window

  df = df.copy()

  nan_indicies = df[names[0]][df[names[0]].isna()].index
  upper_limit = min(nan_indicies[nan_indicies > 5000])
  if combine_str != None:
    lower_limit = max(nan_indicies[nan_indicies < 5000])
  print(lower_limit, upper_limit)

  #print(df.iloc[lower_limit:upper_limit]["total_return_per_day" + combine_str])
  returns = (df.iloc[lower_limit:upper_limit]["total_return_per_day" + combine_str].rolling(window=intervall)).apply(np.prod, raw=True).dropna()#.iloc[::intervall]
  print(np.mean(returns), np.std(returns))
  cost_rent = 1.03 ** (intervall/252)
  return np.mean(returns - cost_rent) / np.std(returns - cost_rent)

In [ ]:
print(finance_data)
print(finance_names)

      Unnamed: 0   day    gurkor   guitars  slingshots     stocks     sugar  \
0              0     1  6.154653  2.794285    2.136536  10.653684  3.324896   
1              1     2  6.189623  2.843068    2.113582  10.674465  3.355736   
2              2     3  6.168641  2.839644    2.116336  10.822372  3.336461   
3              3     4  6.156401  2.823384    2.096137  10.848804  3.288274   
4              4     5  6.124929  2.832798    2.099810  10.860731  3.303694   
...          ...   ...       ...       ...         ...        ...       ...   
5451        5451  5452       NaN       NaN         NaN        NaN       NaN   
5452        5452  5453       NaN       NaN         NaN        NaN       NaN   
5453        5453  5454       NaN       NaN         NaN        NaN       NaN   
5454        5454  5455       NaN       NaN         NaN        NaN       NaN   
5455        5455  5456       NaN       NaN         NaN        NaN       NaN   

         water  tranquillity  
0     3.896149      

In [ ]:
for name in finance_names:
  finance_data = remove_1000(finance_data, name)

[ 6.31581555  6.825179    9.27880519  9.05006504 11.37014512] [ 6.26452498  6.85266821  9.181506    9.04200502 11.46444389]
[2.83043642 5.74900107 4.85888389 6.81445677 5.05114593] [2.85406559 5.41987344 4.85304529 6.82222064 4.90162681]
[2.05786636 3.39560267 3.81753475 4.92829802 3.06994825] [2.07016982 3.44239554 3.88330984 4.92515297 3.00913931]
[9.10307757 8.68991977 6.27277201 5.13249983 4.63075894] [8.83745388 8.8367195  6.20489274 5.0972318  4.54038977]
[3.57234686 3.39488235 2.07917243 3.30055146 1.760932  ] [3.64148906 3.23149229 2.07161867 3.30825403 1.72400296]
[3.97410653 4.69767736 5.89565424 6.21961548 6.6929798 ] [3.99621039 4.704141   5.88280296 6.20774392 6.7114342 ]
[ 7.46344565 10.34647171 10.41322314 18.65861411 10.20343293] [ 7.40623013 10.29561348 10.40686586 19.16719644  9.89192626]


In [ ]:
periods = [[4, 15],
           [7, 25],
           [14, 50],
           [25, 100],
           [50, 200]]

In [ ]:
finance_data_buy = finance_data.copy()

for i, name in enumerate(finance_names):
  for short_window, long_window in periods:
    finance_data_buy = moving_avarage_cross_over(finance_data_buy, name, short_window, long_window)
  #finance_data_buy = moving_avarage_cross_over(finance_data_buy, name, short_window = 4, long_window = 15)
  #finance_data_buy = moving_avarage_cross_over(finance_data_buy, name, short_window = 7, long_window = 25)
  #finance_data_buy = moving_avarage_cross_over(finance_data_buy, name, short_window = 14, long_window = 50)
  #finance_data_buy = moving_avarage_cross_over(finance_data_buy, name, short_window = 25, long_window = 100)
  #finance_data_buy = moving_avarage_cross_over(finance_data_buy, name, short_window = 50, long_window = 200)


In [ ]:
params_grid = []
count = 0
for k in [50]:#[1, 5, 10]:
  for d in [2]:#[1, 2, 3]:
    for x in [0.005, 0.05]:#[0.001, 0.005, 0.01, 0.05]:
      for c in [0.01, 0.05]:#[0.005, 0.01, 0.05, 0.1]:
        for j in [5, 8]:#, 20, 25]:
          params_grid.append([k, d, x, c, j])
          count += 1
print(count)

8


In [ ]:

for i in range(len(params_grid)):
  for j, name in enumerate(finance_names):

    print(i)
    print(params_grid[i])
    combine_str = "_" + "_".join([str(j) for j in params_grid[i]])
    finance_data_buy = channal_breakout(finance_data_buy, name, x = params_grid[i][2], d = params_grid[i][1], j = params_grid[i][4], c = params_grid[i][3], k = params_grid[i][0], combine_str=combine_str)

0
[50, 2, 0.005, 0.01, 5]
793 897
0
[50, 2, 0.005, 0.01, 5]
1232 1400
0
[50, 2, 0.005, 0.01, 5]
1287 1309
0
[50, 2, 0.005, 0.01, 5]
776 612
0
[50, 2, 0.005, 0.01, 5]
1397 1154
0
[50, 2, 0.005, 0.01, 5]
685 592
0
[50, 2, 0.005, 0.01, 5]
1094 2267
1
[50, 2, 0.005, 0.01, 8]
495 669
1
[50, 2, 0.005, 0.01, 8]
395 280
1
[50, 2, 0.005, 0.01, 8]
219 372
1
[50, 2, 0.005, 0.01, 8]
51 0
1
[50, 2, 0.005, 0.01, 8]
51 153
1
[50, 2, 0.005, 0.01, 8]
587 732
1
[50, 2, 0.005, 0.01, 8]
436 282
2
[50, 2, 0.005, 0.05, 5]
0 0
2
[50, 2, 0.005, 0.05, 5]
1189 1387
2
[50, 2, 0.005, 0.05, 5]
1154 1028
2
[50, 2, 0.005, 0.05, 5]
1520 2565
2
[50, 2, 0.005, 0.05, 5]
1200 1642
2
[50, 2, 0.005, 0.05, 5]
0 0
2
[50, 2, 0.005, 0.05, 5]
891 1452
3
[50, 2, 0.005, 0.05, 8]
0 0
3
[50, 2, 0.005, 0.05, 8]
1045 1711
3
[50, 2, 0.005, 0.05, 8]
988 1625
3
[50, 2, 0.005, 0.05, 8]
1899 2509
3
[50, 2, 0.005, 0.05, 8]
1433 2066
3
[50, 2, 0.005, 0.05, 8]
0 0
3
[50, 2, 0.005, 0.05, 8]
942 1658
4
[50, 2, 0.05, 0.01, 5]
0 0
4
[50, 2, 0.05

In [ ]:
print(finance_data_buy)

      Unnamed: 0   day    gurkor   guitars  slingshots     stocks     sugar  \
0              0     1  6.154653  2.794285    2.136536  10.653684  3.324896   
1              1     2  6.189623  2.843068    2.113582  10.674465  3.355736   
2              2     3  6.168641  2.839644    2.116336  10.822372  3.336461   
3              3     4  6.156401  2.823384    2.096137  10.848804  3.288274   
4              4     5  6.124929  2.832798    2.099810  10.860731  3.303694   
...          ...   ...       ...       ...         ...        ...       ...   
5451        5451  5452       NaN       NaN         NaN        NaN       NaN   
5452        5452  5453       NaN       NaN         NaN        NaN       NaN   
5453        5453  5454       NaN       NaN         NaN        NaN       NaN   
5454        5454  5455       NaN       NaN         NaN        NaN       NaN   
5455        5455  5456       NaN       NaN         NaN        NaN       NaN   

         water  tranquillity  gurkor_signal_4_15  .

In [ ]:
for short_window, long_window in periods:
  long_window_text = "_" + str(long_window)
  short_window_text = "_" + str(short_window)
  combine_str = short_window_text + long_window_text
  finance_data_buy = get_return_portfolie(finance_data_buy, finance_names, combine_str = combine_str)
  print(f"short window {short_window}, long window {long_window} checked")

short window 4, long window 15 checked
short window 7, long window 25 checked
short window 14, long window 50 checked
short window 25, long window 100 checked
short window 50, long window 200 checked


In [ ]:
for i in range(len(params_grid)):
    print(i)
    print(params_grid[i])
    combine_str = "_" + "_".join([str(j) for j in params_grid[i]])
    finance_data_buy = get_return_portfolie(finance_data_buy, finance_names, False, combine_str= combine_str, lend_possible = True)
    print(combine_str," checked")

0
[50, 2, 0.005, 0.01, 5]
_50_2_0.005_0.01_5  checked
1
[50, 2, 0.005, 0.01, 8]
_50_2_0.005_0.01_8  checked
2
[50, 2, 0.005, 0.05, 5]
_50_2_0.005_0.05_5  checked
3
[50, 2, 0.005, 0.05, 8]
_50_2_0.005_0.05_8  checked
4
[50, 2, 0.05, 0.01, 5]
_50_2_0.05_0.01_5  checked
5
[50, 2, 0.05, 0.01, 8]
_50_2_0.05_0.01_8  checked
6
[50, 2, 0.05, 0.05, 5]


/tmp/ipykernel_2727/2382786547.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name + "_change" + combine_str] = df[name]/df[name].shift(1)
/tmp/ipykernel_2727/2382786547.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name + "_change" + combine_str] = df[name]/df[name].shift(1)
/tmp/ipykernel_2727/2382786547.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axi

_50_2_0.05_0.05_5  checked
7
[50, 2, 0.05, 0.05, 8]


/tmp/ipykernel_2727/2382786547.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name + "_change" + combine_str] = df[name]/df[name].shift(1)
/tmp/ipykernel_2727/2382786547.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name + "_change" + combine_str] = df[name]/df[name].shift(1)
/tmp/ipykernel_2727/2382786547.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axi

_50_2_0.05_0.05_8  checked


In [ ]:
sharp_ratios_2 = {}
for averaging_time in [1, 5, 21, 63, 252]:
  print(f"Averaging time: {averaging_time}\n\t\tshort,  long,   sharpratio")
  for i in range(len(params_grid)):
    print(i)
    combine_str = "_" + "_".join([str(j) for j in params_grid[i]])
    print(combine_str," checked")
    sharp_ratios_2[combine_str] = get_sharpe_ratio(finance_data_buy, finance_names, averaging_time, combine_str = combine_str)
    print(f"\t\t "+combine_str+f": \t{sharp_ratios_2[combine_str]:4f}")
print("\n", sharp_ratios_2)

Averaging time: 1
		short,  long,   sharpratio
0
_50_2_0.005_0.01_5  checked
247 5256
1.0002320062660313 0.00827811433947272
		 _50_2_0.005_0.01_5: 	0.013856
1
_50_2_0.005_0.01_8  checked
247 5256
1.0001404505415228 0.004309093255922443
		 _50_2_0.005_0.01_8: 	0.005372
2
_50_2_0.005_0.05_5  checked
247 5256
0.9999638897836165 0.011818216018397007
		 _50_2_0.005_0.05_5: 	-0.012981
3
_50_2_0.005_0.05_8  checked
247 5256
1.0001342050049762 0.01072560718934694
		 _50_2_0.005_0.05_8: 	0.001576
4
_50_2_0.05_0.01_5  checked
247 5256
1.0001173037138344 1.354472090042691e-14
		 _50_2_0.05_0.01_5: 	 nan
5
_50_2_0.05_0.01_8  checked
247 5256
1.0001173037138344 1.354472090042691e-14
		 _50_2_0.05_0.01_8: 	 nan
6
_50_2_0.05_0.05_5  checked
247 5256
1.0001173037138344 1.354472090042691e-14
		 _50_2_0.05_0.05_5: 	 nan
7
_50_2_0.05_0.05_8  checked
247 5256
1.0001173037138344 1.354472090042691e-14
		 _50_2_0.05_0.05_8: 	 nan
Averaging time: 5
		short,  long,   sharpratio
0
_50_2_0.005_0.01_5  checked
2

/tmp/ipykernel_2727/2382786547.py:112: RuntimeWarning: invalid value encountered in scalar divide
  return np.mean(returns - cost_rent) / np.std(returns - cost_rent)


247 5256
0.9998096237964129 0.02572763954498904
		 _50_2_0.005_0.05_5: 	-0.030202
3
_50_2_0.005_0.05_8  checked
247 5256
1.0006613259670891 0.024200740341866055
		 _50_2_0.005_0.05_8: 	0.003085
4
_50_2_0.05_0.01_5  checked
247 5256
1.0005866561869274 2.5757174171303632e-14
		 _50_2_0.05_0.01_5: 	-inf
5
_50_2_0.05_0.01_8  checked
247 5256
1.0005866561869274 2.5757174171303632e-14
		 _50_2_0.05_0.01_8: 	-inf
6
_50_2_0.05_0.05_5  checked
247 5256
1.0005866561869274 2.5757174171303632e-14
		 _50_2_0.05_0.05_5: 	-inf
7
_50_2_0.05_0.05_8  checked
247 5256


/tmp/ipykernel_2727/2382786547.py:112: RuntimeWarning: divide by zero encountered in scalar divide
  return np.mean(returns - cost_rent) / np.std(returns - cost_rent)


1.0005866561869274 2.5757174171303632e-14
		 _50_2_0.05_0.05_8: 	-inf
Averaging time: 21
		short,  long,   sharpratio
0
_50_2_0.005_0.01_5  checked
247 5256
1.0048432047051248 0.03666374314495092
		 _50_2_0.005_0.01_5: 	0.064831
1
_50_2_0.005_0.01_8  checked
247 5256
1.0028279741965007 0.022360762534957977
		 _50_2_0.005_0.01_8: 	0.016176
2
_50_2_0.005_0.05_5  checked
247 5256
0.9987240467673694 0.05303055550421627
		 _50_2_0.005_0.05_5: 	-0.070567
3
_50_2_0.005_0.05_8  checked
247 5256
1.0021942315002283 0.04962808717764808
		 _50_2_0.005_0.05_8: 	-0.005482
4
_50_2_0.05_0.01_5  checked
247 5256
1.0024662697723024 8.659739592076221e-14
		 _50_2_0.05_0.01_5: 	-inf
5
_50_2_0.05_0.01_8  checked
247 5256
1.0024662697723024 8.659739592076221e-14
		 _50_2_0.05_0.01_8: 	-inf
6
_50_2_0.05_0.05_5  checked
247 5256
1.0024662697723024 8.659739592076221e-14
		 _50_2_0.05_0.05_5: 	-inf
7
_50_2_0.05_0.05_8  checked
247 5256
1.0024662697723024 8.659739592076221e-14
		 _50_2_0.05_0.05_8: 	-inf
Averagi

In [ ]:
print(finance_data_buy)

      Unnamed: 0   day    gurkor   guitars  slingshots     stocks     sugar  \
0              0     1  6.154653  2.794285    2.136536  10.653684  3.324896   
1              1     2  6.189623  2.843068    2.113582  10.674465  3.355736   
2              2     3  6.168641  2.839644    2.116336  10.822372  3.336461   
3              3     4  6.156401  2.823384    2.096137  10.848804  3.288274   
4              4     5  6.124929  2.832798    2.099810  10.860731  3.303694   
...          ...   ...       ...       ...         ...        ...       ...   
5451        5451  5452       NaN       NaN         NaN        NaN       NaN   
5452        5452  5453       NaN       NaN         NaN        NaN       NaN   
5453        5453  5454       NaN       NaN         NaN        NaN       NaN   
5454        5454  5455       NaN       NaN         NaN        NaN       NaN   
5455        5455  5456       NaN       NaN         NaN        NaN       NaN   

         water  tranquillity  gurkor_signal_4_15  .

In [ ]:
finance_data_buy = get_return_portfolie_buy_and_hold(finance_data_buy, finance_names, weighting="active")

In [ ]:
for averaging_time in [1, 5, 21, 63, 252]:
  print(finance_data_buy.keys()[2:].to_list())
  print(get_sharpe_ratio(finance_data_buy, finance_names, averaging_time, combine_str = "_buy_and_hold"))

['gurkor', 'guitars', 'slingshots', 'stocks', 'sugar', 'water', 'tranquillity', 'gurkor_signal_4_15', 'gurkor_signal_7_25', 'gurkor_signal_14_50', 'gurkor_signal_25_100', 'gurkor_signal_50_200', 'guitars_signal_4_15', 'guitars_signal_7_25', 'guitars_signal_14_50', 'guitars_signal_25_100', 'guitars_signal_50_200', 'slingshots_signal_4_15', 'slingshots_signal_7_25', 'slingshots_signal_14_50', 'slingshots_signal_25_100', 'slingshots_signal_50_200', 'stocks_signal_4_15', 'stocks_signal_7_25', 'stocks_signal_14_50', 'stocks_signal_25_100', 'stocks_signal_50_200', 'sugar_signal_4_15', 'sugar_signal_7_25', 'sugar_signal_14_50', 'sugar_signal_25_100', 'sugar_signal_50_200', 'water_signal_4_15', 'water_signal_7_25', 'water_signal_14_50', 'water_signal_25_100', 'water_signal_50_200', 'tranquillity_signal_4_15', 'tranquillity_signal_7_25', 'tranquillity_signal_14_50', 'tranquillity_signal_25_100', 'tranquillity_signal_50_200', 'gurkor_change_4_15', 'guitars_change_4_15', 'slingshots_change_4_15',

In [ ]:
finance_data_buy = get_return_portfolie_buy_and_hold(finance_data_buy, finance_names, weighting="equal")

In [ ]:
print(finance_data_buy)

      Unnamed: 0   day    gurkor   guitars  slingshots     stocks     sugar  \
0              0     1  6.154653  2.794285    2.136536  10.653684  3.324896   
1              1     2  6.189623  2.843068    2.113582  10.674465  3.355736   
2              2     3  6.168641  2.839644    2.116336  10.822372  3.336461   
3              3     4  6.156401  2.823384    2.096137  10.848804  3.288274   
4              4     5  6.124929  2.832798    2.099810  10.860731  3.303694   
...          ...   ...       ...       ...         ...        ...       ...   
5451        5451  5452       NaN       NaN         NaN        NaN       NaN   
5452        5452  5453       NaN       NaN         NaN        NaN       NaN   
5453        5453  5454       NaN       NaN         NaN        NaN       NaN   
5454        5454  5455       NaN       NaN         NaN        NaN       NaN   
5455        5455  5456       NaN       NaN         NaN        NaN       NaN   

         water  tranquillity  gurkor_signal_4_15  .

In [ ]:
for averaging_time in [1, 5, 21, 63, 252]:
  print(finance_data_buy.keys()[2:].to_list())
  print(get_sharpe_ratio(finance_data_buy, finance_names, averaging_time, combine_str = "_buy_and_hold"))

['gurkor', 'guitars', 'slingshots', 'stocks', 'sugar', 'water', 'tranquillity', 'gurkor_signal_4_15', 'gurkor_signal_7_25', 'gurkor_signal_14_50', 'gurkor_signal_25_100', 'gurkor_signal_50_200', 'guitars_signal_4_15', 'guitars_signal_7_25', 'guitars_signal_14_50', 'guitars_signal_25_100', 'guitars_signal_50_200', 'slingshots_signal_4_15', 'slingshots_signal_7_25', 'slingshots_signal_14_50', 'slingshots_signal_25_100', 'slingshots_signal_50_200', 'stocks_signal_4_15', 'stocks_signal_7_25', 'stocks_signal_14_50', 'stocks_signal_25_100', 'stocks_signal_50_200', 'sugar_signal_4_15', 'sugar_signal_7_25', 'sugar_signal_14_50', 'sugar_signal_25_100', 'sugar_signal_50_200', 'water_signal_4_15', 'water_signal_7_25', 'water_signal_14_50', 'water_signal_25_100', 'water_signal_50_200', 'tranquillity_signal_4_15', 'tranquillity_signal_7_25', 'tranquillity_signal_14_50', 'tranquillity_signal_25_100', 'tranquillity_signal_50_200', 'gurkor_change_4_15', 'guitars_change_4_15', 'slingshots_change_4_15',

In [ ]:
sharp_ratios = {}
for averaging_time in [1, 5, 21, 63, 252]:
  print(f"Averaging time: {averaging_time}\n\t\tshort,  long,   sharpratio")
  for short_window, long_window in periods:
    sharp_ratios[str(averaging_time) + f"_{short_window}_{long_window}"] = get_sharpe_ratio(finance_data_buy, finance_names, averaging_time, short_window=short_window, long_window = long_window)
    print(f"\t\t{short_window}, \t{long_window}: \t{sharp_ratios[str(averaging_time) + f"_{short_window}_{long_window}"]:4f}")
print("\n", sharp_ratios)

Averaging time: 1
		short,  long,   sharpratio
247 5256
247          NaN
248          NaN
249     0.993350
250     0.999181
251     0.997611
          ...   
5251    1.003575
5252    0.992058
5253    1.006518
5254    0.993149
5255    1.004019
Name: total_return_per_day_4_15, Length: 5009, dtype: float64
1.0000983376617676 0.006851374711672563
		4, 	15: 	-0.002768
247 5256
247          NaN
248          NaN
249     0.993350
250     0.999181
251     1.000640
          ...   
5251    1.006765
5252    0.992995
5253    0.992073
5254    0.994029
5255    0.996857
Name: total_return_per_day_7_25, Length: 5009, dtype: float64
1.000207390201076 0.006774786398690723
		7, 	25: 	0.013297
247 5256
247          NaN
248          NaN
249     0.995306
250     0.999534
251     1.002426
          ...   
5251    1.006765
5252    0.992995
5253    0.992073
5254    0.994029
5255    0.989631
Name: total_return_per_day_14_50, Length: 5009, dtype: float64
1.0002935652402452 0.006276966364460393
		14, 	50: 	0.0280

In [ ]:
print(get_sharpe_ratio(finance_data_buy, finance_names, short_window=14, long_window = 50))
print(get_sharpe_ratio(finance_data_buy, finance_names, short_window=25, long_window=100))
print(get_sharpe_ratio(finance_data_buy, finance_names, short_window=50, long_window=200))

247 5256
247          NaN
248          NaN
249     0.995306
250     0.999534
251     1.002426
          ...   
5251    1.006765
5252    0.992995
5253    0.992073
5254    0.994029
5255    0.989631
Name: total_return_per_day_14_50, Length: 5009, dtype: float64
1.0058757675172763 0.02779391412431211
0.12267065839388956
247 5256
247          NaN
248          NaN
249     0.993088
250     0.996686
251     1.003033
          ...   
5251    1.002469
5252    0.995013
5253    0.995096
5254    0.995570
5255    0.993239
Name: total_return_per_day_25_100, Length: 5009, dtype: float64
1.0057501039257424 0.029681498991708244
0.11063572477778248
247 5256
247          NaN
248          NaN
249     0.993262
250     0.995719
251     1.005293
          ...   
5251    1.002469
5252    0.995013
5253    0.995096
5254    0.995570
5255    0.993239
Name: total_return_per_day_50_200, Length: 5009, dtype: float64
1.0045568028507206 0.029393679939237083
0.07112185622005386


In [ ]:
print(get_sharpe_ratio(finance_data_buy, finance_names, 252, short_window=14, long_window = 50))
print(get_sharpe_ratio(finance_data_buy, finance_names, 252, short_window=25, long_window=100))
print(get_sharpe_ratio(finance_data_buy, finance_names, 252, short_window=50, long_window=200))

247 5256
247          NaN
248          NaN
249     0.995306
250     0.999534
251     1.002426
          ...   
5251    1.006765
5252    0.992995
5253    0.992073
5254    0.994029
5255    0.989631
Name: total_return_per_day_14_50, Length: 5009, dtype: float64
1.0661574633935589 0.09422964602613895
0.38371643021484864
247 5256
247          NaN
248          NaN
249     0.993088
250     0.996686
251     1.003033
          ...   
5251    1.002469
5252    0.995013
5253    0.995096
5254    0.995570
5255    0.993239
Name: total_return_per_day_25_100, Length: 5009, dtype: float64
1.0599607436889489 0.10954322563331934
0.27350612980156425
247 5256
247          NaN
248          NaN
249     0.993262
250     0.995719
251     1.005293
          ...   
5251    1.002469
5252    0.995013
5253    0.995096
5254    0.995570
5255    0.993239
Name: total_return_per_day_50_200, Length: 5009, dtype: float64
1.043082803417262 0.09782947483359708
0.13373069250872666


# Extra

In [ ]:
import torch
from torch import nn, optim
from torch.amp import GradScaler, autocast
import torch.nn.functional as F

In [ ]:
torch.backends.cudnn.benchmark = True

torch.backends.cudnn.deterministic = False

torch.set_default_dtype(torch.float32)

print(f"Using device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
print(f"cudnn.benchmark enabled: {torch.backends.cudnn.benchmark}")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


Using device: cpu
cudnn.benchmark enabled: True


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Using device: cpu


In [ ]:
class LSTM6(nn.Module):
    def __init__(self, input_size=2, size_hidden=8, num_layers=3,
                 dropout_layers_lstm=0.2, dropout_out=0.3):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=size_hidden,
            num_layers=num_layers,
            dropout=dropout_layers_lstm if num_layers > 1 else 0.0,
            batch_first=True
        )

        self.eps = 1e-7
        self.norm = nn.LayerNorm(size_hidden, eps=self.eps)
        self.fc = nn.Linear(size_hidden, 1)
        self.dropout = nn.Dropout(dropout_out)

        self.num_layers = num_layers
        self.size_hidden = size_hidden
        self.hidden_state = None

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self._init_weights()

    def _init_weights(self):
        for name, param in self.named_parameters():
            if 'weight' in name and len(param.shape) > 1:
                if 'lstm' in name:
                    nn.init.xavier_uniform_(param)
                elif 'fc' in name:
                    nn.init.xavier_uniform_(param, gain=0.1)
            elif 'bias' in name:
                nn.init.zeros_(param)

    def forward(self, x):
        # Safety check for NaN inputs
        if torch.isnan(x).any():
            print("Warning: NaN in input!")
            return torch.zeros(x.size(0), 1, device=x.device)

        x = torch.clamp(x, min=-10, max=10)

        if self.hidden_state is None:
            self.hidden_state = self.init_hidden(x.size(0), x.device)
        else:
            h, c = self.hidden_state
            if h.size(1) != x.size(0):
                self.hidden_state = self.init_hidden(x.size(0), x.device)

        out, hidden_new = self.lstm(x, self.hidden_state)
        out = torch.clamp(out, min=-100, max=100)

        out = self.norm(out)
        out = out[:, -1, :]

        out = self.dropout(out)
        out = self.fc(out)

        out = torch.clamp(out, min=-50, max=50)
        self.hidden_state = (hidden_new[0].detach(), hidden_new[1].detach())

        return out

    def init_hidden(self, batch_size, device=None):
        if device is None:
            device = next(self.parameters()).device

        h0 = torch.randn(self.num_layers, batch_size, self.size_hidden, device=device) * 0.01
        c0 = torch.randn(self.num_layers, batch_size, self.size_hidden, device=device) * 0.01

        self.hidden_state = (h0, c0)
        return (h0, c0)

    def reset_hidden(self):
        self.hidden_state = None

In [ ]:
def make_windows(data_x, labels_y, seq_len, delta_init=0):
    T = data_x.shape[0]
    if T < seq_len + delta_init:
        raise ValueError("I think")
    starts = torch.arange(delta_init, T - seq_len + 1)
    x_windows = torch.stack([data_x[i:i + seq_len] for i in starts])
    y_windows = torch.stack([labels_y[i + seq_len - 1] for i in starts]) # minus one for correctness
    return x_windows, y_windows


In [ ]:
import copy

In [ ]:
def train_model(model, data_x, data_y, data_x_val, data_y_val, freq, num_epochs=100, learning_rate=1e-4, seq_len=5, warmup_days=10, decay=0.01):
    model.to(device)

    all_windows_x = []
    all_windows_y = []
    for stock in range(len(data_x)):
        window_x, window_y = make_windows(data_x[stock], data_y[stock], seq_len)
        if len(window_x) > warmup_days:
            all_windows_x.append(window_x.to(device, non_blocking=True))
            all_windows_y.append(window_y.to(device, non_blocking=True).float())

    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=decay)
    scaler = GradScaler()
    pos_weight = torch.tensor([(1 - freq) / freq], dtype=torch.float32, device=device)

    best_val_loss = float('inf')
    best_model = None

    for epoch in range(num_epochs):
        model.train()

        stock_order = torch.randperm(len(all_windows_x), device=device)

        epoch_loss = 0.0
        epoch_samples = 0

        for stock_pos in range(len(stock_order)):
            stock_idx = stock_order[stock_pos].item()

            window_x = all_windows_x[stock_idx]
            window_y = all_windows_y[stock_idx]

            model.reset_hidden()

            with torch.no_grad():
                for i in range(0, warmup_days, 16):
                    batch_end = min(i + 16, warmup_days)
                    _ = model(window_x[i:batch_end])

            logits = torch.empty(len(window_x)-warmup_days, device=device)
            targets = window_y[warmup_days:]

            pos = 0
            batch_size = 32
            for i in range(warmup_days, len(window_x), batch_size):
                batch_end = min(i + batch_size, len(window_x))
                data = window_x[i:batch_end]

                with autocast(device_type='cuda'):
                    batch_logits = model(data).squeeze(1)

                batch_len = batch_logits.shape[0]
                logits[pos:pos + batch_len] = batch_logits
                pos += batch_len

            optimizer.zero_grad(set_to_none=True)

            with autocast(device_type='cuda'):
                loss = F.binary_cross_entropy_with_logits(
                    logits,
                    targets,
                    pos_weight=pos_weight,
                    reduction='mean'
                )

            scaler.scale(loss).backward()

            epoch_loss += loss.item() * logits.shape[0]
            epoch_samples += logits.shape[0]

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)

            scaler.step(optimizer)
            scaler.update()


            if (stock_pos + 1) % 10 == 0:
                print(f"\t\tStock {stock_pos + 1}/{len(all_windows_x)}")

        avg_epoch_loss = epoch_loss / epoch_samples

        val_loss = evaluate_model(model, data_x_val, data_y_val, freq, seq_len, warmup_days)
        print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {avg_epoch_loss:.4}, Val Loss: {val_loss:.4}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model = copy.deepcopy(model.state_dict())

    if best_model is not None:
        model.load_state_dict(best_model)

    return model


def evaluate_model(model, data_x_val, data_y_val, freq, seq_len, warmup_days):
    model.eval()
    total_loss = 0.0
    total_samples = 0

    pos_weight = torch.tensor([(1 - freq) / freq], dtype=torch.float32, device=device)

    with torch.no_grad():
        for stock_idx in range(len(data_x_val)):
            window_x, window_y = make_windows(data_x_val[stock_idx], data_y_val[stock_idx], seq_len)

            if len(window_x) <= warmup_days:
                continue
            window_x = window_x.to(device, non_blocking=True)
            window_y = window_y.to(device, non_blocking=True).float()

            model.reset_hidden()

            # Warmup
            for i in range(0, warmup_days, 32):
                batch_end = min(i + 32, warmup_days)
                _ = model(window_x[i:batch_end])

            batch_size = 64
            for i in range(warmup_days, len(window_x), batch_size):
                batch_end = min(i + batch_size, len(window_x))

                logits = model(window_x[i:batch_end])
                logits = logits.squeeze(1)

                loss = F.binary_cross_entropy_with_logits(
                    logits, window_y[i:batch_end],
                    pos_weight=pos_weight,
                    reduction='sum'
                )

                total_loss += loss.item()
                total_samples += (batch_end - i)

    return total_loss / total_samples if total_samples > 0 else float('inf')

In [ ]:
def predict_avg_return_on_investment(model, stock_params_normilized, stock_params_non_normilized, threshold=0.5, batch_size=1, seq_len=10):
  predictions = np.zeros(stock_params_non_normilized.size(0) - 11)
  model.eval()
  with torch.no_grad():
            window_x, window_y = make_windows(stock_params_normilized, torch.zeros(stock_params_normilized.size(0)), seq_len, delta_init=0)
            window_x = window_x.to(device, non_blocking=True)
            window_y = window_y.to(device, non_blocking=True).float()
            model.eval()
            model.reset_hidden()

            for i in range(0, 10, 10):
                _ = model(window_x[0:10])

            batch_size = 32  # Large batch for evaluation
            for i in range(0, len(window_x), batch_size):
                batch_end = min(10+ i + batch_size, len(window_x)-1)

                logits = model(window_x[10+i:batch_end])
                logits = logits.squeeze(1)

                predictions[i:batch_end-10] = logits.cpu().numpy().reshape(-1,)
  return_on_investment = ((stock_params_non_normilized[11:,0])/(stock_params_non_normilized[10:-1,0]) - 1).numpy()

  decisions = (1/(1+np.exp(-predictions)) > threshold).astype(int)
  profit_sum = np.sum(return_on_investment[decisions == 1])
  returns =  np.where(decisions == 1, return_on_investment, 1.03**(1/252))
  return profit_sum, np.shape(return_on_investment[decisions == 1])[0], returns

In [ ]:

def combine_params(json_data, stock_name, stock_name2, after_split=0):
    adj_close = json_data.get(stock_name).to_numpy()
    target = json_data.get(stock_name + "_t").to_numpy()
    adj_close2 = json_data.get(stock_name2).to_numpy()
    target2 = json_data.get(stock_name2 + "_t").to_numpy()

    nan_idx = np.where(np.isnan(adj_close))[0]
    nan_idx_t = np.where(np.isnan(target))[0]
    nan_idx_2 = np.where(np.isnan(adj_close))[0]
    nan_idx_t_2 = np.where(np.isnan(target))[0]

    nan_idx = np.concatenate((nan_idx, nan_idx_t))
    nan_idx = np.unique(nan_idx)

    if len(nan_idx) == 0 and len(nan_idx_2) == 0:
      pass
    elif len(nan_idx) == 0:
      i2_min = min(nan_idx_2)
      i2_max = max(nan_idx_2)
      intervalls = [[0, i2_min], [(i2_max + 1), len(adj_close)]]
      adj_close = adj_close[intervalls[after_split][0]:intervalls[after_split][1]]
      target = target[intervalls[after_split][0]:intervalls[after_split][1]]
      adj_close2 = adj_close2[intervalls[after_split][0]:intervalls[after_split][1]]
      target2 = target2[intervalls[after_split][0]:intervalls[after_split][1]]
    elif len(nan_idx_2) == 0:
      i1_min = min(nan_idx)
      i1_max = max(nan_idx)
      intervalls = [[0, i1_min], [(i1_max + 1) , len(adj_close)]]
      adj_close = adj_close[intervalls[after_split][0]:intervalls[after_split][1]]
      target = target[intervalls[after_split][0]:intervalls[after_split][1]]
      adj_close2 = adj_close2[intervalls[after_split][0]:intervalls[after_split][1]]
      target2 = target2[intervalls[after_split][0]:intervalls[after_split][1]]
    else:
      i1_min = min(nan_idx)
      i1_max = max(nan_idx)

      i2_min = min(nan_idx_2)
      i2_max = max(nan_idx_2)
      if i1_min > i2_min:
          i1_min, i2_min = i2_min, i1_min
          i1_max, i2_max = i2_max, i1_max

      l0 = i1_min - 0
      l1 = i2_min - (i1_max + 1)
      l2 = len(adj_close) - (i2_max + 1)
      m = min(l0, l1, l2)
      if l0 == m:
        intervalls = [[(i1_max + 1), i2_min], [(i2_max + 1) , len(adj_close)]]
      elif l1 == m:
        intervalls = [[0, i1_min], [(i2_max + 1) , len(adj_close)]]
      else:
        intervalls = [[i1_min, (i1_max + 1)], [0, i1_min]]
      adj_close = adj_close[intervalls[after_split][0]:intervalls[after_split][1]]
      target = target[intervalls[after_split][0]:intervalls[after_split][1]]
      adj_close2 = adj_close2[intervalls[after_split][0]:intervalls[after_split][1]]
      target2 = target2[intervalls[after_split][0]:intervalls[after_split][1]]

    size_volume = len(adj_close)
    x_data = np.zeros((size_volume, 2), dtype=np.float64)
    y_data = np.zeros(size_volume, dtype=np.int64)
    x_data[:, 0] = np.array(adj_close)
    x_data[:, 1] = np.array(adj_close2)

    y_data = np.where((target > 1.005) & (target > target2), 1, 0).astype(int)
    print(y_data.shape, x_data.shape)
    print("\t", stock_name, x_data.shape, y_data.shape, y_data.sum())
    return x_data, y_data


In [ ]:
def normalize_one_stock(price_volume, lookback_days=20):

    n_days = price_volume.shape[0] - 1
    normalized = torch.zeros_like(price_volume[:-1])

    for today in range(n_days):
            normalized[today, 0] = price_volume[max(today, 1), 0]/(price_volume[max(today-1, 0), 0] + 1e-8) - 1
    return normalized

In [ ]:
for name in finance_names:
  finance_data[name+"_t"] = (finance_data[name].shift(-1) / finance_data[name] ).astype(float)
training_data, test_data =  training_test_split(finance_data, part_slit=0.8)
training_data, valid_data =  training_test_split(training_data, part_slit=0.8)

Index([1398, 1399, 1400, 1401, 1402, 1403, 1404, 1405, 1406, 1407,
       ...
       5446, 5447, 5448, 5449, 5450, 5451, 5452, 5453, 5454, 5455],
      dtype='int64', length=250)
0 5255
Index([1398, 1399, 1400, 1401, 1402, 1403, 1404, 1405, 1406, 1407, 1408, 1409,
       1410, 1411, 1412, 1413, 1414, 1415, 1416, 1417, 1418, 1419, 1420, 1421,
       1422, 1423, 1424, 1425, 1426, 1427, 1428, 1429, 1430, 1431, 1432, 1433,
       1434, 1435, 1436, 1437, 1438, 1439, 1440, 1441, 1442, 1443, 1444, 1445,
       1446, 1447],
      dtype='int64')
0 4204


In [ ]:
print(test_data)

      Unnamed: 0   day     gurkor   guitars  slingshots    stocks     sugar  \
4204        4204  4205  11.157629  4.930754    3.084341  4.509189  1.854322   
4205        4205  4206  11.238734  4.863387    3.046735  4.401822  1.852360   
4206        4206  4207  11.243079  4.836440    3.007309  4.469612  1.830775   
4207        4207  4208  11.283631  4.715679    2.872653  4.524837  1.834700   
4208        4208  4209  11.277838  4.742626    2.901768  4.518365  1.789568   
...          ...   ...        ...       ...         ...       ...       ...   
5451        5451  5452        NaN       NaN         NaN       NaN       NaN   
5452        5452  5453        NaN       NaN         NaN       NaN       NaN   
5453        5453  5454        NaN       NaN         NaN       NaN       NaN   
5454        5454  5455        NaN       NaN         NaN       NaN       NaN   
5455        5455  5456        NaN       NaN         NaN       NaN       NaN   

         water  tranquillity  gurkor_t  guitars_t  

In [ ]:
num_stocks = len(finance_names)
print(num_stocks)
stocks_name = finance_names

#creating torch tensors
train_data_input = [[] for i in range(num_stocks * (num_stocks-1) * 2)]
val_data_input = [[] for i in range(num_stocks * (num_stocks-1) * 2)]
test_data_input = [[] for i in range(num_stocks * (num_stocks-1) * 2)]

train_data_target = [[] for i in range(num_stocks * (num_stocks-1) * 2)]
val_data_target = [[] for i in range(num_stocks * (num_stocks-1) * 2)]
test_data_target = [[] for i in range(num_stocks * (num_stocks-1) * 2)]

for i, stock_name in enumerate(stocks_name):
  stocks2 = stocks_name.copy()
  stocks2.remove(stock_name)
  for j, stock_name2 in enumerate(stocks2):
    for k in range(2):
      train_data_input[2 * (i * (num_stocks - 1) + j) + k] = torch.tensor(combine_params(training_data, stock_name, stock_name2, k)[0], dtype=torch.float32)
      val_data_input[2 * (i * (num_stocks - 1) + j) + k] = torch.tensor(combine_params(valid_data, stock_name, stock_name2, 0)[0], dtype=torch.float32)
      test_data_input[2 * (i * (num_stocks - 1) + j) + k] = torch.tensor(combine_params(test_data, stock_name, stock_name2, 0)[0], dtype=torch.float32)


      train_data_target[2 * (i * (num_stocks - 1) + j) + k] = torch.tensor(combine_params(training_data, stock_name, stock_name2, k)[1], dtype=torch.float32)
      val_data_target[2 * (i * (num_stocks - 1) + j) + k] = torch.tensor(combine_params(valid_data, stock_name2, stock_name)[1], dtype=torch.float32)
      test_data_target[2 * (i * (num_stocks - 1) + j) + k]  = torch.tensor(combine_params(test_data, stock_name2, stock_name)[1], dtype=torch.float32)

      print(2 * (i * (num_stocks - 1) + j) + k)

7
(197,) (197, 2)
	 gurkor (197, 2) (197,) 11
(841,) (841, 2)
	 gurkor (841, 2) (841,) 91
(1051,) (1051, 2)
	 gurkor (1051, 2) (1051,) 80
(197,) (197, 2)
	 gurkor (197, 2) (197,) 11
(841,) (841, 2)
	 guitars (841, 2) (841,) 261
(1051,) (1051, 2)
	 guitars (1051, 2) (1051,) 340
0
(3115,) (3115, 2)
	 gurkor (3115, 2) (3115,) 190
(841,) (841, 2)
	 gurkor (841, 2) (841,) 91
(1051,) (1051, 2)
	 gurkor (1051, 2) (1051,) 80
(3115,) (3115, 2)
	 gurkor (3115, 2) (3115,) 190
(841,) (841, 2)
	 guitars (841, 2) (841,) 261
(1051,) (1051, 2)
	 guitars (1051, 2) (1051,) 340
1
(197,) (197, 2)
	 gurkor (197, 2) (197,) 13
(841,) (841, 2)
	 gurkor (841, 2) (841,) 85
(1051,) (1051, 2)
	 gurkor (1051, 2) (1051,) 77
(197,) (197, 2)
	 gurkor (197, 2) (197,) 13
(841,) (841, 2)
	 slingshots (841, 2) (841,) 291
(1051,) (1051, 2)
	 slingshots (1051, 2) (1051,) 379
2
(3115,) (3115, 2)
	 gurkor (3115, 2) (3115,) 191
(841,) (841, 2)
	 gurkor (841, 2) (841,) 85
(1051,) (1051, 2)
	 gurkor (1051, 2) (1051,) 77
(3115,)

In [ ]:
# for inputs because raw inputs is very very very bad compared to also normilized
train_data_normilized = [normalize_one_stock(stock_serie).detach() for stock_serie in train_data_input]
val_data_normilized   = [normalize_one_stock(stock_serie).detach() for stock_serie in val_data_input]
test_data_normilized  = [normalize_one_stock(stock_serie).detach() for stock_serie in test_data_input]

In [ ]:
print(test_data_input, "\n", test_data_normilized)

[tensor([[11.1576,  4.9308],
        [11.2387,  4.8634],
        [11.2431,  4.8364],
        ...,
        [13.7986,  8.1341],
        [13.8003,  8.1915],
        [13.7736,  8.1238]]), tensor([[11.1576,  4.9308],
        [11.2387,  4.8634],
        [11.2431,  4.8364],
        ...,
        [13.7986,  8.1341],
        [13.8003,  8.1915],
        [13.7736,  8.1238]]), tensor([[11.1576,  3.0843],
        [11.2387,  3.0467],
        [11.2431,  3.0073],
        ...,
        [13.7986,  6.4060],
        [13.8003,  6.4443],
        [13.7736,  6.3710]]), tensor([[11.1576,  3.0843],
        [11.2387,  3.0467],
        [11.2431,  3.0073],
        ...,
        [13.7986,  6.4060],
        [13.8003,  6.4443],
        [13.7736,  6.3710]]), tensor([[11.1576,  4.5092],
        [11.2387,  4.4018],
        [11.2431,  4.4696],
        ...,
        [13.7986,  6.6779],
        [13.8003,  6.6556],
        [13.7736,  6.5933]]), tensor([[11.1576,  4.5092],
        [11.2387,  4.4018],
        [11.2431,  4.4696],


In [ ]:
freq = np.array([(subpart.numpy()).sum() for subpart in train_data_target]).sum() / np.sum(np.array([(subpart.numpy()).shape[0] for subpart in train_data_target]))

print(freq)
model = LSTM6(input_size=2, size_hidden=12, dropout_layers_lstm=0., dropout_out=0.05, num_layers=1).to(device)

# compile it
if hasattr(torch, 'compile'):
    model = torch.compile(model)

0.2139406487232574


In [ ]:
gamma = 2 #decrease the freq of hotspot (higher weight on wrong hotspotatos
train_model(model, train_data_normilized, train_data_target, val_data_normilized, val_data_target, freq/gamma, num_epochs=20, learning_rate=0.001, seq_len = 10, decay=0.001)

/tmp/ipykernel_30499/2407525012.py:13: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = GradScaler()
/tmp/ipykernel_30499/2407525012.py:49: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with autocast(device_type='cuda'):
/tmp/ipykernel_30499/2407525012.py:49: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with autocast(device_type='cuda'):
/tmp/ipykernel_30499/2407525012.py:49: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with autocast(device_type='cuda'):
/tmp/ipykernel_30499/2407525012.py:58: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with autocast(device_type='cuda'):
/tmp/ipykernel_30499/2407525012.py:58: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with autocast(device_type='cuda'):
/tmp/ipykernel_30499/2407525012.py:49: UserWarning: CUDA is no

		Stock 10/84
		Stock 20/84
		Stock 30/84
		Stock 40/84
		Stock 50/84
		Stock 60/84
		Stock 70/84
		Stock 80/84
Epoch 1/20, Loss: 1.633, Val Loss: 1.629


/tmp/ipykernel_30499/2407525012.py:49: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with autocast(device_type='cuda'):
/tmp/ipykernel_30499/2407525012.py:58: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with autocast(device_type='cuda'):


		Stock 10/84
		Stock 20/84
		Stock 30/84
		Stock 40/84
		Stock 50/84
		Stock 60/84
		Stock 70/84
		Stock 80/84
Epoch 2/20, Loss: 1.592, Val Loss: 1.631
		Stock 10/84
		Stock 20/84
		Stock 30/84
		Stock 40/84
		Stock 50/84
		Stock 60/84
		Stock 70/84
		Stock 80/84
Epoch 3/20, Loss: 1.589, Val Loss: 1.62
		Stock 10/84
		Stock 20/84
		Stock 30/84
		Stock 40/84
		Stock 50/84
		Stock 60/84
		Stock 70/84
		Stock 80/84
Epoch 4/20, Loss: 1.586, Val Loss: 1.622
		Stock 10/84
		Stock 20/84
		Stock 30/84
		Stock 40/84
		Stock 50/84
		Stock 60/84
		Stock 70/84
		Stock 80/84
Epoch 5/20, Loss: 1.589, Val Loss: 1.625
		Stock 10/84
		Stock 20/84
		Stock 30/84
		Stock 40/84
		Stock 50/84
		Stock 60/84
		Stock 70/84
		Stock 80/84
Epoch 6/20, Loss: 1.589, Val Loss: 1.62
		Stock 10/84
		Stock 20/84
		Stock 30/84
		Stock 40/84
		Stock 50/84
		Stock 60/84
		Stock 70/84
		Stock 80/84
Epoch 7/20, Loss: 1.586, Val Loss: 1.622
		Stock 10/84
		Stock 20/84
		Stock 30/84
		Stock 40/84
		Stock 50/84
		Stock 60/84


OptimizedModule(
  (_orig_mod): LSTM6(
    (lstm): LSTM(2, 12, batch_first=True)
    (norm): LayerNorm((12,), eps=1e-07, elementwise_affine=True)
    (fc): Linear(in_features=12, out_features=1, bias=True)
    (dropout): Dropout(p=0.05, inplace=False)
  )
)

In [ ]:
for theta in np.arange(0.69, 0.7, 0.0001):
  total_profit = 0
  total_days= 0
  total_updates = 0
  avg_profit = 0
  for i in range(num_stocks):
    sum_profit, updates, _ = predict_avg_return_on_investment(model, val_data_normilized[i], val_data_input[i], threshold=theta, batch_size=1, seq_len=10)
    total_profit += sum_profit

    avg_profit += sum_profit/(val_data_normilized[i].size(0)-9)

    total_days += val_data_normilized[i].size(0)-9
    total_updates += updates

  avg_profit_per_stock = avg_profit/num_stocks
  ROI_day = total_profit/total_days
  ROI_trade = total_profit/total_updates

  print(f"{theta:.4f}\tROI_per day   \t\t", ROI_day)
  print(f"{theta:.4f}\tROI_per trade \t\t", ROI_trade)
  print("ROI_per day averaging stock \t",   avg_profit_per_stock)

0.6900	ROI_per day   		 0.00024975877
0.6900	ROI_per trade 		 0.0002531092
ROI_per day averaging stock 	 0.00024975874
0.6901	ROI_per day   		 0.00024975877
0.6901	ROI_per trade 		 0.0002531092
ROI_per day averaging stock 	 0.00024975874
0.6902	ROI_per day   		 0.00024975877
0.6902	ROI_per trade 		 0.0002531092
ROI_per day averaging stock 	 0.00024975874
0.6903	ROI_per day   		 0.00024975877
0.6903	ROI_per trade 		 0.0002531092
ROI_per day averaging stock 	 0.00024975874
0.6904	ROI_per day   		 0.00024975877
0.6904	ROI_per trade 		 0.0002531092
ROI_per day averaging stock 	 0.00024975874
0.6905	ROI_per day   		 0.00024975877
0.6905	ROI_per trade 		 0.0002531092
ROI_per day averaging stock 	 0.00024975874
0.6906	ROI_per day   		 0.00024975877
0.6906	ROI_per trade 		 0.0002531092
ROI_per day averaging stock 	 0.00024975874
0.6907	ROI_per day   		 0.00024975877
0.6907	ROI_per trade 		 0.0002531092
ROI_per day averaging stock 	 0.00024975874
0.6908	ROI_per day   		 0.00024975877
0.6908	ROI

/tmp/ipykernel_30499/253134971.py:17: RuntimeWarning: invalid value encountered in scalar divide
  ROI_trade = total_profit/total_updates


0.6929	ROI_per day   		 0.0
0.6929	ROI_per trade 		 nan
ROI_per day averaging stock 	 0.0
0.6930	ROI_per day   		 0.0
0.6930	ROI_per trade 		 nan
ROI_per day averaging stock 	 0.0
0.6931	ROI_per day   		 0.0
0.6931	ROI_per trade 		 nan
ROI_per day averaging stock 	 0.0
0.6932	ROI_per day   		 0.0
0.6932	ROI_per trade 		 nan
ROI_per day averaging stock 	 0.0
0.6933	ROI_per day   		 0.0
0.6933	ROI_per trade 		 nan
ROI_per day averaging stock 	 0.0
0.6934	ROI_per day   		 0.0
0.6934	ROI_per trade 		 nan
ROI_per day averaging stock 	 0.0
0.6935	ROI_per day   		 0.0
0.6935	ROI_per trade 		 nan
ROI_per day averaging stock 	 0.0
0.6936	ROI_per day   		 0.0
0.6936	ROI_per trade 		 nan
ROI_per day averaging stock 	 0.0
0.6937	ROI_per day   		 0.0
0.6937	ROI_per trade 		 nan
ROI_per day averaging stock 	 0.0
0.6938	ROI_per day   		 0.0
0.6938	ROI_per trade 		 nan
ROI_per day averaging stock 	 0.0
0.6939	ROI_per day   		 0.0
0.6939	ROI_per trade 		 nan
ROI_per day averaging stock 	 0.0
0.6940	ROI

KeyboardInterrupt: 

In [ ]:
ROI_day_per_company = [0,0]
ROI_day = [0,0]
ROI_trade = [0,0]
Trade_freq = [0,0]
for idx, theta in enumerate([0, 0.6927]):
  total_profit = 0
  total_days= 0
  total_updates = 0
  avg_profit = 0
  returns_all = np.array([])
  for i in range(num_stocks):
    sum_profit, updates, returns = predict_avg_return_on_investment(model, test_data_normilized[i], test_data_input[i], threshold=theta, batch_size=1, seq_len=10)
    total_profit += sum_profit.sum()
    avg_profit += sum_profit/(test_data_normilized[i].size(0)-10)
    total_days += test_data_normilized[i].size(0)-10
    total_updates += updates
    returns_all = np.concatenate((returns_all, returns))

  print(np.mean(returns_all - ((1.03)**(1/252)-1))/np.std(returns_all - ((1.03)**(1/252)-1)))

  ROI_day[idx] = total_profit/total_days

  ROI_day_per_company[idx] = avg_profit/num_stocks

  ROI_trade[idx] = total_profit/total_updates
  Trade_freq[idx] = total_updates/total_days

print("Test data day:")
print(f"           \t    return per day     return per trade    return per day per stock      trading freq")
print(f"invest all days  \t{ROI_day[0]*100:.3f} % \t{ROI_trade[0]*100:.3f} % \t\t {ROI_day_per_company[0]*100:.3f}% \t\t{Trade_freq[0]:.2f}x")
print(f"invest with rules\t{ROI_day[1]*100:.3f} % \t{ROI_trade[1]*100:.3f} % \t\t {ROI_day_per_company[1]*100:.3f}% \t\t{Trade_freq[1]:.2f}x")
days = val_data_normilized[i].size(0)+1

print("\n\nTest data (yearly returns) :")
print(f"           \t    return per year     return per trade    return per day per stock      trading freq")
print(f"invest all days  \t{(1+ROI_day[0])**days *100-100:.3f} % \t{(1+ROI_trade[0])**days *100-100:.3f} % \t\t {(1+ROI_day_per_company[0])**days *100-100:.3f}% \t\t{Trade_freq[0]:.2f}x")
print(f"invest with rules\t{(1+ROI_day[1])**days *100-100:.3f} % \t{(1+ ROI_trade[1])**days *100-100:.3f} % \t\t {(1+ ROI_day_per_company[1])**days *100-100:.3f}% \t\t{Trade_freq[1]:.2f}x")

0.027682533586499303
0.14776123494099633
Test data day:
           	    return per day     return per trade    return per day per stock      trading freq
invest all days  	0.022 % 	0.022 % 		 0.022% 		1.00x
invest with rules	0.023 % 	0.023 % 		 0.023% 		0.98x


Test data (yearly returns) :
           	    return per year     return per trade    return per day per stock      trading freq
invest all days  	20.207 % 	20.207 % 		 20.207% 		1.00x
invest with rules	21.102 % 	21.613 % 		 21.102% 		0.98x
